# MimicMotion · Google Colab 單 GPU 安裝與執行

> 本專案 fork 自 [Tencent/MimicMotion](https://github.com/Tencent/MimicMotion)；本 fork 新增的功能由 ChatGPT 完成。

請在 Colab 的「執行階段 → 變更執行階段類型」選擇 **GPU**，然後從上到下執行。開始前，在 [SVD 模型頁面](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt-1-1)取得使用權限，並準備 Hugging Face 讀取 token。安裝和模型下載需要磁碟空間與網路時間。完整說明請見 [COLAB.md](https://github.com/delvilo/MimicMotion/blob/main/COLAB.md)。


In [ ]:
from pathlib import Path
import subprocess

PROJECT = Path('/content/MimicMotion')
subprocess.run(['nvidia-smi'], check=True)
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                    'https://github.com/delvilo/MimicMotion.git', str(PROJECT)], check=True)
else:
    if not (PROJECT / '.git').exists():
        raise RuntimeError(f'{PROJECT} 已存在且不是 Git 倉庫，請先處理該目錄。')
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote.rstrip('/').removesuffix('.git') != 'https://github.com/delvilo/MimicMotion':
        raise RuntimeError(f'現有 Git 倉庫並非 delvilo/MimicMotion：{remote}')
    branch = subprocess.check_output(['git', '-C', str(PROJECT), 'branch', '--show-current'], text=True).strip()
    if branch != 'main':
        raise RuntimeError(f'預期 main 分支，實際為 {branch!r}；請自行處理現有分支。')
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', 'main'], check=True)
print('專案路徑：', PROJECT)


## 取得 Hugging Face 存取權

請先在瀏覽器取得 SVD 存取權。以下儲存格優先讀取 Colab Secrets 裡的 `HF_TOKEN`；若沒有，會在隱藏輸入框要求 token。它只供安裝器下載模型，不會寫進筆記本。


In [ ]:
import getpass
import os

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('Hugging Face 讀取 token（輸入內容不顯示）：')
if not token:
    raise ValueError('請先提供 HF_TOKEN 並取得 SVD 模型存取權。')
os.environ['HF_TOKEN'] = token
del token
print('HF_TOKEN 已暫時提供給安裝程序。')


## 安裝依賴及模型

安裝器會依 Colab GPU 驅動選擇 PyTorch CUDA wheel，建立兩個獨立的 Python 環境，下載及驗證模型。完成後會建立單 GPU 啟動器。若遇到授權或磁碟空間錯誤，請處理後重跑本儲存格。


In [ ]:
try:
    subprocess.run(['bash', str(PROJECT / 'scripts/colab.sh'), '--install-only'],
                   cwd=str(PROJECT), check=True)
finally:
    os.environ.pop('HF_TOKEN', None)


## 選擇素材

可以把素材放到 Google Drive，或先上傳到 `/content`。若要掛載 Drive，執行下面儲存格並授權。推論時會先將影片與圖片複製到 `/content` 本地磁碟，以便讀取與寫入。


In [ ]:
USE_GOOGLE_DRIVE = True # @param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil

VIDEO_PATH = '/content/drive/MyDrive/MimicMotion/dance.mp4' # @param {type:"string"}
IMAGE_PATH = '/content/drive/MyDrive/MimicMotion/actor.jpg' # @param {type:"string"}
MODE = 'replace' # @param ['replace', 'generate']
TRANSPARENT_BACKGROUND = False # @param {type:"boolean"}
ALPHA_CODEC = 'prores4444' # @param ['prores4444', 'qtrle']
RESOLUTION = 384 # @param {type:"integer"}
NUM_FRAMES = 16 # @param {type:"integer"}
FRAMES_OVERLAP = 4 # @param {type:"integer"}
DECODE_CHUNK_SIZE = 2 # @param {type:"integer"}
SAVE_TO_DRIVE = True # @param {type:"boolean"}

if SAVE_TO_DRIVE and not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('請先掛載 Google Drive，或將 SAVE_TO_DRIVE 設為 False。')

inputs = Path('/content/mimicmotion-inputs')
inputs.mkdir(exist_ok=True)
source_video, source_image = Path(VIDEO_PATH).expanduser(), Path(IMAGE_PATH).expanduser()
for path in (source_video, source_image):
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'請填寫存在且非空的素材檔案路徑：{path}')
video_local = inputs / ('dance' + source_video.suffix)
image_local = inputs / ('actor' + source_image.suffix)
for src, dst in ((source_video, video_local), (source_image, image_local)):
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
output_dir = PROJECT / 'outputs' / 'colab'
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / ('result.mov' if TRANSPARENT_BACKGROUND else 'result.mp4')
print('本地輸入：', video_local, image_local)
print('本地輸出：', output_file)


## 執行推論並保存輸出

`replace` 保留舞蹈影片背景與鏡頭；打開透明背景開關後只輸出有 Alpha 的生成角色。`generate` 沿用整幅生成模式。示範設定以較小解析度和 tile 開始；若 GPU 記憶體不足，請縮短素材或降低解析度。完成後，選擇將影片與 JSON 紀錄複製到 Drive。


In [ ]:
command = ['bash', str(PROJECT / 'scripts/colab.sh'), '--run-only', '--',
           '--ref_video_path', str(video_local),
           '--ref_image_path', str(image_local),
           '--mode', MODE, '--device', 'cuda:0', '--dtype', 'float16',
           '--resolution', str(RESOLUTION),
           '--num_frames', str(NUM_FRAMES),
           '--frames_overlap', str(FRAMES_OVERLAP),
           '--decode_chunk_size', str(DECODE_CHUNK_SIZE),
           '--output_file', str(output_file), '--overwrite']
if TRANSPARENT_BACKGROUND:
    command += ['--transparent_background', '--alpha_codec', ALPHA_CODEC]
subprocess.run(command, cwd=str(PROJECT), check=True)
print('生成完成：', output_file)
if SAVE_TO_DRIVE:
    dest_dir = Path('/content/drive/MyDrive/MimicMotion/outputs')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('請先掛載 Google Drive，或將 SAVE_TO_DRIVE 設為 False。')
    dest_dir.mkdir(parents=True, exist_ok=True)
    for item in (output_file, output_file.with_suffix('.json')):
        if item.is_file():
            shutil.copy2(item, dest_dir / item.name)
    print('已複製至：', dest_dir / output_file.name)
